In [2]:
# ============================================================
# INSTALL + IMPORT ALL REQUIRED LIBRARIES
# ============================================================

# Install required packages
!pip install -q google-genai pypdf python-dotenv transformers torch sentence-transformers faiss-cpu ipywidgets streamlit

# ------------------------------------------------------------
# Python imports
# ------------------------------------------------------------

import os
import re
import csv
import math
import getpass

from datetime import datetime

import numpy as np
import faiss
import torch

# Gemini
from google import genai
from google.genai import types

# PDF
from pypdf import PdfReader

# Embeddings
from sentence_transformers import SentenceTransformer

# Colab
from google.colab import files

# Chat UI
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

print("✅ All libraries installed and imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 120.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 135.4 MB/s eta 0:00:00
✅ All libraries installed and imported successfully!


In [4]:
!pip install -q google-genai pypdf python-dotenv transformers torch sentence-transformers faiss-cpu ipywidgets

In [5]:
import faiss
print("FAISS version:", faiss.__version__)
print("✅ FAISS is working!")

FAISS version: 1.15.1
✅ FAISS is working!


In [7]:
# ============================================================
# MULTI-AGENT RAG ASSISTANT - COMPLETE ONE-CELL COLAB APP
# ============================================================

import sys
import subprocess
import importlib.util
import os
import re
import csv
import time
import math
from datetime import datetime

# ------------------------------------------------------------
# 1. INSTALL REQUIRED PACKAGES
# ------------------------------------------------------------

packages = [
    "google-genai",
    "pypdf",
    "python-dotenv",
    "transformers",
    "torch",
    "sentence-transformers",
    "faiss-cpu",
    "ipywidgets"
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + packages
)

print("✅ Required packages installed/checked.")

# ------------------------------------------------------------
# 2. IMPORTS
# ------------------------------------------------------------

import numpy as np
import faiss
import torch

from google import genai
from google.genai import types

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

from google.colab import files
import ipywidgets as widgets

from IPython.display import display, Markdown, clear_output

print("✅ Imports successful.")
print("FAISS version:", faiss.__version__)

# ------------------------------------------------------------
# 3. GEMINI API SETUP
# ------------------------------------------------------------

API_KEY = os.environ.get("GEMINI_API_KEY")

if not API_KEY:
    import getpass
    API_KEY = getpass.getpass("🔑 Enter your Gemini API Key: ")

if not API_KEY:
    raise RuntimeError("❌ Gemini API key was not provided.")

client = genai.Client(api_key=API_KEY)

MODEL = "gemini-3.8-flash"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

print("✅ Gemini client initialized.")

# ------------------------------------------------------------
# 4. EMBEDDING MODEL
# ------------------------------------------------------------

print("⏳ Loading embedding model...")

embedder = SentenceTransformer(EMBEDDING_MODEL)

print("✅ Embedding model loaded.")

# ------------------------------------------------------------
# 5. GLOBAL DATA
# ------------------------------------------------------------

documents = []
chunks = []
chunk_sources = []

faiss_index = None

conversation_history = []

MAX_HISTORY = 12

last_agent = "NONE"
last_sources = []

# ------------------------------------------------------------
# 6. GOOGLE SEARCH TOOL
# ------------------------------------------------------------

google_search_tool = types.Tool(
    google_search=types.GoogleSearch()
)

# ------------------------------------------------------------
# 7. PDF TEXT EXTRACTION
# ------------------------------------------------------------

def extract_pdf_text(pdf_path):

    text = ""

    try:
        reader = PdfReader(pdf_path)

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    except Exception as e:
        print("⚠️ PDF extraction error:", e)

    return text


# ------------------------------------------------------------
# 8. TEXT CHUNKING
# ------------------------------------------------------------

def create_chunks(text, chunk_size=900, overlap=150):

    words = text.split()

    result = []

    start = 0

    while start < len(words):

        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end])

        if chunk.strip():
            result.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return result


# ------------------------------------------------------------
# 9. PROCESS PDF
# ------------------------------------------------------------

def process_pdf(pdf_path):

    global documents
    global chunks
    global chunk_sources

    filename = os.path.basename(pdf_path)

    text = extract_pdf_text(pdf_path)

    if not text.strip():
        print("⚠️ No readable text:", filename)
        return

    documents.append({
        "name": filename,
        "text": text
    })

    pdf_chunks = create_chunks(text)

    for chunk in pdf_chunks:

        chunks.append(chunk)
        chunk_sources.append(filename)

    print(
        f"📄 {filename} → "
        f"{len(pdf_chunks)} chunks"
    )


# ------------------------------------------------------------
# 10. UPLOAD DOCUMENTS
# ------------------------------------------------------------

print("\n📂 Upload your PDF documents.")

uploaded_files = files.upload()

for filename in uploaded_files:

    if filename.lower().endswith(".pdf"):
        process_pdf(filename)

    else:
        print("⚠️ Skipped non-PDF:", filename)


# ------------------------------------------------------------
# 11. BUILD FAISS INDEX
# ------------------------------------------------------------

if not chunks:

    print(
        "\n⚠️ No PDF chunks found."
        "\nYou can still use Web Search and General AI."
    )

    faiss_index = None

else:

    print("\n⏳ Creating embeddings...")

    embeddings = embedder.encode(
        chunks,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    embeddings = embeddings.astype("float32")

    faiss.normalize_L2(embeddings)

    dimension = embeddings.shape[1]

    faiss_index = faiss.IndexFlatIP(dimension)

    faiss_index.add(embeddings)

    print(
        f"✅ FAISS index created with "
        f"{len(chunks)} chunks."
    )


# ------------------------------------------------------------
# 12. DOCUMENT RETRIEVAL
# ------------------------------------------------------------

def retrieve_documents(query, top_k=7):

    if faiss_index is None or not chunks:
        return []

    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = faiss_index.search(
        query_embedding,
        min(top_k, len(chunks))
    )

    results = []

    for score, index in zip(scores[0], indices[0]):

        if index < 0:
            continue

        results.append({
            "text": chunks[index],
            "source": chunk_sources[index],
            "score": float(score)
        })

    return results


# ------------------------------------------------------------
# 13. SAFE GEMINI CALL
# ------------------------------------------------------------

def safe_gemini_call(
    prompt,
    use_web=False,
    retries=3
):

    for attempt in range(retries):

        try:

            config_kwargs = {}

            if use_web:
                config_kwargs["tools"] = [
                    google_search_tool
                ]

            config = types.GenerateContentConfig(
                **config_kwargs
            )

            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=config
            )

            if getattr(response, "text", None):

                return response.text

        except Exception as e:

            error_text = str(e)

            if (
                "503" in error_text
                or "UNAVAILABLE" in error_text
                or "429" in error_text
                or "RESOURCE_EXHAUSTED" in error_text
            ):

                if attempt < retries - 1:

                    time.sleep(
                        2 ** attempt
                    )

                    continue

            print("⚠️ Gemini error:", error_text)

            return None

    return None


# ------------------------------------------------------------
# 14. LOCAL CALCULATOR
# ------------------------------------------------------------

def calculator_agent(query):

    expression = query.strip()

    expression = expression.replace("^", "**")

    allowed = re.fullmatch(
        r"[0-9+\-*/().%\s\^]+",
        expression
    )

    if not allowed:

        return (
            "I can calculate arithmetic expressions "
            "such as `25*4`, `100/5`, or `(10+5)*2`."
        )

    try:

        result = eval(
            expression,
            {
                "__builtins__": {}
            },
            {}
        )

        return f"**Calculation result:** `{result}`"

    except Exception:

        return "I couldn't calculate that expression."


# ------------------------------------------------------------
# 15. CONVERSATION MEMORY
# ------------------------------------------------------------

app_memory = []


def get_memory():

    if not app_memory:
        return "No previous conversation."

    history = []

    for item in app_memory[-6:]:

        history.append(
            f"User: {item['question']}\n"
            f"Assistant: {item['answer']}"
        )

    return "\n\n".join(history)


# ------------------------------------------------------------
# 16. SUPERVISOR ROUTER
# ------------------------------------------------------------

def supervisor_route(query):

    q = query.lower()

    agents = []

    education_words = [
        "study",
        "learn",
        "learning",
        "exam",
        "examination",
        "concept",
        "explain",
        "education",
        "subject",
        "revision",
        "notes",
        "course",
        "syllabus",
        "tutorial"
    ]

    technical_words = [
        "python",
        "java",
        "programming",
        "code",
        "coding",
        "debug",
        "error",
        "technical",
        "ai",
        "ml",
        "machine learning",
        "deep learning",
        "rag",
        "llm",
        "api",
        "github",
        "project",
        "dataset",
        "model",
        "algorithm",
        "deployment",
        "software"
    ]

    career_words = [
        "career",
        "job",
        "internship",
        "resume",
        "cv",
        "interview",
        "placement",
        "skill gap",
        "skills",
        "ai engineer",
        "software engineer",
        "career path",
        "professional"
    ]

    document_words = [
        "pdf",
        "document",
        "documents",
        "uploaded",
        "file",
        "files",
        "according to",
        "from my document",
        "from the document",
        "in the pdf"
    ]

    web_words = [
        "latest",
        "current",
        "today",
        "recent",
        "news",
        "2026",
        "this week",
        "online"
    ]

    if any(word in q for word in education_words):
        agents.append("EDUCATION")

    if any(word in q for word in technical_words):
        agents.append("TECHNICAL")

    if any(word in q for word in career_words):
        agents.append("CAREER")

    if any(word in q for word in document_words):
        agents.append("DOCUMENT")

    if any(word in q for word in web_words):
        agents.append("WEB")

    # Mathematical query
    if re.fullmatch(
        r"[\d\s+\-*/().%^]+",
        query.strip()
    ):
        agents.append("CALCULATOR")

    # Multi-agent query
    if len(agents) == 0:
        agents.append("GENERAL")

    # If complex query has multiple specialists,
    # also activate document retrieval.
    specialists = {
        "EDUCATION",
        "TECHNICAL",
        "CAREER"
    }

    if (
        len(
            set(agents).intersection(specialists)
        ) >= 2
        and faiss_index is not None
    ):
        if "DOCUMENT" not in agents:
            agents.append("DOCUMENT")

    return list(dict.fromkeys(agents))


# ------------------------------------------------------------
# 17. EDUCATION AGENT
# ------------------------------------------------------------

def education_agent(query, memory):

    prompt = f"""
You are the Education Agent.

Help the student with:
- Study questions
- Concepts and explanations
- Exam preparation
- Learning plans

Use simple student-friendly language.

Previous conversation:
{memory}

Current question:
{query}

Give a practical and clear answer.
"""

    answer = safe_gemini_call(prompt)

    if answer:
        return answer

    return (
        "### Education Agent\n"
        "I can help with concepts, study plans, "
        "exam preparation, revision, and learning paths."
    )


# ------------------------------------------------------------
# 18. TECHNICAL AGENT
# ------------------------------------------------------------

def technical_agent(query, memory):

    prompt = f"""
You are the Technical Agent.

Help with:
- Programming
- Python / Java
- AI and Machine Learning
- RAG and LLMs
- Project development
- Technical troubleshooting
- GitHub and deployment

Previous conversation:
{memory}

Current question:
{query}

Explain step-by-step when useful.
Use beginner-friendly language.
"""

    answer = safe_gemini_call(prompt)

    if answer:
        return answer

    return (
        "### Technical Agent\n"
        "I can help with programming, AI/ML, "
        "projects, debugging, RAG, GitHub, and deployment."
    )


# ------------------------------------------------------------
# 19. CAREER AGENT
# ------------------------------------------------------------

def career_agent(query, memory):

    prompt = f"""
You are the Career Agent.

Help with:
- Career paths
- Skill-gap analysis
- Resume and project suggestions
- Internship preparation
- Interview preparation
- AI Engineer preparation

Previous conversation:
{memory}

Current question:
{query}

Give practical steps and examples.
"""

    answer = safe_gemini_call(prompt)

    if answer:
        return answer

    return (
        "### Career Agent\n"
        "I can help with career paths, skill gaps, "
        "projects, resumes, internships, and interviews."
    )


# ------------------------------------------------------------
# 20. DOCUMENT RAG AGENT
# ------------------------------------------------------------

def document_agent(query):

    results = retrieve_documents(
        query,
        top_k=7
    )

    if not results:

        return (
            "I couldn't find this in the uploaded "
            "documents."
        ), []

    # Filter weak matches
    useful_results = [
        r for r in results
        if r["score"] >= 0.25
    ]

    if not useful_results:

        return (
            "I couldn't find sufficiently relevant "
            "information in the uploaded documents."
        ), results

    context_parts = []

    for r in useful_results:

        context_parts.append(
            f"""
SOURCE: {r['source']}
SIMILARITY: {r['score']:.3f}

{r['text']}
"""
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are the Document RAG Agent.

Answer the user's question using the retrieved
document context below.

Important:
- Do not invent information.
- If the documents do not contain the answer,
  clearly say that.
- Mention relevant source names when useful.

User question:
{query}

Retrieved context:
{context}
"""

    answer = safe_gemini_call(prompt)

    if not answer:

        answer = (
            "I found relevant information in the "
            "uploaded documents, but the language "
            "model is temporarily unavailable."
        )

    return answer, useful_results


# ------------------------------------------------------------
# 21. WEB SEARCH AGENT
# ------------------------------------------------------------

def web_agent(query):

    prompt = f"""
You are the Web Search Agent.

Answer the user's question using current web
information when appropriate.

Question:
{query}

Clearly distinguish current information from
general knowledge.
"""

    answer = safe_gemini_call(
        prompt,
        use_web=True
    )

    if answer:
        return answer

    return (
        "Web search is temporarily unavailable. "
        "Please try again."
    )


# ------------------------------------------------------------
# 22. GENERAL AI AGENT
# ------------------------------------------------------------

def general_agent(query, memory):

    prompt = f"""
You are the General AI Agent.

Answer the user's question clearly and accurately.

Previous conversation:
{memory}

Question:
{query}
"""

    answer = safe_gemini_call(prompt)

    if answer:
        return answer

    return (
        "I can help with questions about studies, "
        "AI/ML, programming, projects, careers, "
        "documents, and general topics."
    )


# ------------------------------------------------------------
# 23. FINAL SYNTHESIS AGENT
# ------------------------------------------------------------

def synthesis_agent(
    query,
    specialist_outputs,
    memory
):

    combined = "\n\n".join(
        [
            f"### {name} Agent\n{answer}"
            for name, answer in specialist_outputs.items()
        ]
    )

    prompt = f"""
You are the Final Synthesis Agent.

Combine the useful information from the specialist
agents into ONE clear answer.

Rules:
- Do not mention internal agent instructions.
- Do not repeat the same point unnecessarily.
- Preserve important technical details.
- Use headings and bullet points when useful.
- Use previous conversation context.
- If document information is provided, do not invent
  facts beyond the retrieved context.

Previous conversation:
{memory}

User question:
{query}

Specialist responses:
{combined}

Write the final answer.
"""

    answer = safe_gemini_call(prompt)

    if answer:
        return answer

    # Local fallback
    return combined


# ------------------------------------------------------------
# 24. MAIN MULTI-AGENT SYSTEM
# ------------------------------------------------------------

def run_multi_agent(query):

    agents = supervisor_route(query)

    memory = get_memory()

    activity = []

    activity.append(
        "🧠 Supervisor: Query analyzed"
    )

    activity.append(
        "🤖 Agents activated: "
        + ", ".join(agents)
    )

    specialist_outputs = {}

    source_results = []

    # Education
    if "EDUCATION" in agents:

        activity.append(
            "📚 Education Agent: Activated"
        )

        specialist_outputs[
            "Education"
        ] = education_agent(
            query,
            memory
        )

    # Technical
    if "TECHNICAL" in agents:

        activity.append(
            "💻 Technical Agent: Activated"
        )

        specialist_outputs[
            "Technical"
        ] = technical_agent(
            query,
            memory
        )

    # Career
    if "CAREER" in agents:

        activity.append(
            "🎯 Career Agent: Activated"
        )

        specialist_outputs[
            "Career"
        ] = career_agent(
            query,
            memory
        )

    # Documents
    if "DOCUMENT" in agents:

        activity.append(
            "📚 RAG: Retrieving document chunks"
        )

        answer, source_results = document_agent(
            query
        )

        specialist_outputs[
            "Document RAG"
        ] = answer

        activity.append(
            f"📄 RAG: {len(source_results)} "
            "relevant chunks retrieved"
        )

    # Web
    if "WEB" in agents:

        activity.append(
            "🌐 Web Search: Activated"
        )

        specialist_outputs[
            "Web Search"
        ] = web_agent(query)

    # Calculator
    if "CALCULATOR" in agents:

        activity.append(
            "🧮 Calculator: Activated"
        )

        specialist_outputs[
            "Calculator"
        ] = calculator_agent(query)

    # General
    if "GENERAL" in agents:

        activity.append(
            "💬 General AI: Activated"
        )

        specialist_outputs[
            "General AI"
        ] = general_agent(
            query,
            memory
        )

    # Synthesis
    if len(specialist_outputs) > 1:

        activity.append(
            "🔄 Synthesis: Combining responses"
        )

        final_answer = synthesis_agent(
            query,
            specialist_outputs,
            memory
        )

    else:

        final_answer = next(
            iter(specialist_outputs.values())
        )

    activity.append(
        "✅ Final answer generated"
    )

    # Save memory
    app_memory.append({
        "question": query,
        "answer": final_answer
    })

    if len(app_memory) > MAX_HISTORY:
        del app_memory[:-MAX_HISTORY]

    return (
        final_answer,
        agents,
        activity,
        source_results
    )


# ------------------------------------------------------------
# 25. USER INTERFACE
# ------------------------------------------------------------

app_title = widgets.HTML(
    """
    <div style="
        padding:18px;
        border-radius:15px;
        background:#111827;
        color:white;
        margin-bottom:12px;
    ">
        <h1 style="margin:0;">
        🤖 Multi-Agent RAG Assistant
        </h1>

        <p style="margin:8px 0 0 0;">
        Education • Technical • Career • RAG •
        Web Search • Calculator
        </p>
    </div>
    """
)

app_input = widgets.Textarea(
    placeholder=(
        "Ask something...\n\n"
        "Example: I'm an AI/ML student. "
        "What should I learn to become an AI engineer "
        "and what projects should I build?"
    ),
    layout=widgets.Layout(
        width="100%",
        height="90px"
    )
)

app_send = widgets.Button(
    description="🚀 Ask",
    button_style="primary",
    layout=widgets.Layout(
        width="120px"
    )
)

app_clear = widgets.Button(
    description="🗑 Clear",
    layout=widgets.Layout(
        width="120px"
    )
)

app_save = widgets.Button(
    description="💾 Save Chat",
    layout=widgets.Layout(
        width="120px"
    )
)

app_output = widgets.Output(
    layout=widgets.Layout(
        width="100%"
    )
)

app_activity = widgets.Output(
    layout=widgets.Layout(
        width="100%"
    )
)

app_sources = widgets.Output(
    layout=widgets.Layout(
        width="100%"
    )
)

# ------------------------------------------------------------
# 26. QUICK PROMPTS
# ------------------------------------------------------------

quick1 = widgets.Button(
    description="📚 Study Plan"
)

quick2 = widgets.Button(
    description="💻 AI/ML Project"
)

quick3 = widgets.Button(
    description="🎯 Career Path"
)

quick4 = widgets.Button(
    description="📄 Ask Documents"
)

def quick_prompt(text):

    def handler(_):

        app_input.value = text

        send_message()

    return handler


quick1.on_click(
    quick_prompt(
        "Create a study plan for learning AI and Machine Learning."
    )
)

quick2.on_click(
    quick_prompt(
        "Suggest AI/ML projects that I can build as a student."
    )
)

quick3.on_click(
    quick_prompt(
        "What skills should I develop for an AI Engineer career?"
    )
)

quick4.on_click(
    quick_prompt(
        "What information can you find in my uploaded documents?"
    )
)


# ------------------------------------------------------------
# 27. SEND MESSAGE
# ------------------------------------------------------------

def send_message(_=None):

    query = app_input.value.strip()

    if not query:
        return

    app_input.value = ""

    with app_output:

        clear_output(wait=True)

        display(
            Markdown(
                f"### 👤 You\n{query}"
            )
        )

        display(
            Markdown(
                "⏳ **Agents are working...**"
            )
        )

    with app_activity:

        clear_output(wait=True)

        display(
            Markdown(
                "### 🔎 Agent Activity"
            )
        )

        display(
            Markdown(
                "Supervisor: Analyzing query..."
            )
        )

    (
        answer,
        agents,
        activity,
        source_results
    ) = run_multi_agent(query)

    with app_output:

        clear_output(wait=True)

        display(
            Markdown(
                f"### 👤 You\n{query}"
            )
        )

        display(
            Markdown(
                "### 🤖 Assistant\n\n"
                + answer
            )
        )

    with app_activity:

        clear_output(wait=True)

        display(
            Markdown(
                "### 🔎 Agent Activity\n\n"
                + "\n\n".join(
                    [
                        f"- {x}"
                        for x in activity
                    ]
                )
            )
        )

    with app_sources:

        clear_output(wait=True)

        display(
            Markdown(
                "### 📚 Sources / Retrieved Chunks"
            )
        )

        if source_results:

            for i, result in enumerate(
                source_results,
                start=1
            ):

                source_name = result["source"]

                score = result["score"]

                snippet = result["text"]

                if len(snippet) > 700:
                    snippet = snippet[:700] + "..."

                display(
                    Markdown(
                        f"""
**{i}. {source_name}**

Similarity score: `{score:.3f}`

> {snippet}
"""
                    )
                )

        else:

            display(
                Markdown(
                    "_No document chunks were used for this query._"
                )
            )


# ------------------------------------------------------------
# 28. CLEAR CHAT
# ------------------------------------------------------------

def clear_chat(_=None):

    app_memory.clear()

    with app_output:
        clear_output()

    with app_activity:
        clear_output()

    with app_sources:
        clear_output()

    app_input.value = ""

    print("🗑 Conversation memory cleared.")


# ------------------------------------------------------------
# 29. SAVE CHAT
# ------------------------------------------------------------

def save_chat(_=None):

    if not app_memory:

        print("⚠️ No conversation to save.")

        return

    filename = (
        "multi_agent_rag_chat_"
        + datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )
        + ".csv"
    )

    with open(
        filename,
        "w",
        newline="",
        encoding="utf-8"
    ) as file:

        writer = csv.writer(file)

        writer.writerow([
            "Question",
            "Answer"
        ])

        for item in app_memory:

            writer.writerow([
                item["question"],
                item["answer"]
            ])

    files.download(filename)


# ------------------------------------------------------------
# 30. BUTTON EVENTS
# ------------------------------------------------------------

app_send.on_click(send_message)

app_clear.on_click(clear_chat)

app_save.on_click(save_chat)

app_send.on_click(send_message)


# ------------------------------------------------------------
# 31. DISPLAY APP
# ------------------------------------------------------------

quick_box = widgets.HBox(
    [
        quick1,
        quick2,
        quick3,
        quick4
    ],
    layout=widgets.Layout(
        flex_flow="row wrap"
    )
)

button_box = widgets.HBox(
    [
        app_send,
        app_clear,
        app_save
    ]
)

display(
    app_title
)

display(
    widgets.HTML(
        "<b>Quick Prompts</b>"
    )
)

display(
    quick_box
)

display(
    widgets.HTML(
        "<br><b>Your Question</b>"
    )
)

display(
    app_input
)

display(
    button_box
)

display(
    widgets.HTML(
        "<br>"
    )
)

display(
    app_output
)

display(
    app_activity
)

display(
    app_sources
)

print("\n" + "=" * 60)
print("🎉 MULTI-AGENT RAG ASSISTANT IS READY!")
print("=" * 60)

print(
    "\nArchitecture:"
)

print(
    "USER → SUPERVISOR → "
    "SPECIALIST AGENTS → RAG/TOOLS → "
    "SYNTHESIS → GEMINI → ANSWER"
)

print(
    "\nAgents:"
)

print(
    "📚 Education | 💻 Technical | "
    "🎯 Career | 📄 Document RAG | "
    "🌐 Web | 🧮 Calculator | 💬 General"
)

print(
    "\nTry:"
)

print(
    "I'm an AI/ML student. "
    "What should I learn to become an AI engineer "
    "and what projects should I build?"
)

✅ Required packages installed/checked.
✅ Imports successful.
FAISS version: 1.15.1
🔑 Enter your Gemini API Key: ··········
✅ Gemini client initialized.
⏳ Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model loaded.

📂 Upload your PDF documents.


Saving technical_support.pdf--.pdf to technical_support.pdf-- (1).pdf
📄 technical_support.pdf-- (1).pdf → 2 chunks

⏳ Creating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

HTML(value='<br>')

Output(layout=Layout(width='100%'))

Output(layout=Layout(width='100%'))

Output(layout=Layout(width='100%'))


🎉 MULTI-AGENT RAG ASSISTANT IS READY!

Architecture:
USER → SUPERVISOR → SPECIALIST AGENTS → RAG/TOOLS → SYNTHESIS → GEMINI → ANSWER

Agents:
📚 Education | 💻 Technical | 🎯 Career | 📄 Document RAG | 🌐 Web | 🧮 Calculator | 💬 General

Try:
I'm an AI/ML student. What should I learn to become an AI engineer and what projects should I build?
